In [0]:
from pyspark.sql.functions import (
    to_timestamp, substring, lpad, concat,
    year, month, dayofmonth, hour, make_timestamp, 
    col, count, when, isnan, avg, sum as spark_sum, lit, desc,
    min, max, mean, stddev, log1p, broadcast,
    regexp_replace, try_to_timestamp, date_trunc, expr, least
)
# from pyspark.sql.functions import to_timestamp# , col, date_trunc, hour, expr, regexp_replace, try_to_timestamp
import pyspark.sql.functions as F
from pyspark.sql.window import Window
import matplotlib.pyplot as plt
import pandas as pd
from pyspark.sql import DataFrame
from graphframes import GraphFrame
from functools import reduce

# Adding to avoid casting errors when converting weather string columns to numeric
spark.conf.set("spark.sql.ansi.enabled", "false")

In [0]:
# Always read from parquet
TEAM_DIR = "dbfs:/student-groups/Group_3_2"

otpw_train_df = spark.read.parquet(f"{TEAM_DIR}/041226_otpw_train_df.parquet")
otpw_test_df = spark.read.parquet(f"{TEAM_DIR}/041226_otpw_test_df.parquet")

# Destination congestion features

We currently have `delayed_flights_2_4hr_before` and `delay_pct_flights_2_4hr_before` computed at the origin airport. We need the same features for the destination airport. for exmaple, for each flight, count how many flights arriving at the destination airport were delayed in the 2-4 hours before this flight's scheduled departure. This captures arrival slot conflicts and gate congestion at the destination. We could use same window logic as the existing origin congestion features, but group by DEST instead of ORIGIN. Name them `dest_delayed_flights_2_4hr_before` and `dest_delay_pct_flights_2_4hr_before`.

In [0]:
def compute_window_features_dest(otpw_df):
    '''Compute features describing inbound traffic conditions at the destination airport
    in the 2-4 hours before each flight's scheduled departure. For each flight, we count:

        1. The number of flights that ARRIVED at the destination airport between 2 to 4 hrs before
        and were delayed (ARR_DEL15 = 1).

        2. The total number of flights that arrived in that window.

    These features use only arrival timestamps that occur 2 hours before.
    '''
    arrivals = otpw_df.select(
        F.col("DEST").alias("ARR_AIRPORT"),
        F.col("arrive_date_time").alias("ARR_TIME"),
        F.col("ARR_DEL15")
    )

    # For a flight departing at time T, we want arrivals 2 to 4 hrs before.
    # Shifting Arrival time forward by 2 hours
    arrivals_shifted = arrivals.withColumn(
        "ARR_TIME_PLUS_2H", F.col("ARR_TIME") + F.expr("INTERVAL 2 HOURS")
    )

    joined = otpw_df.join(
        arrivals_shifted,
        (otpw_df.DEST == arrivals_shifted.ARR_AIRPORT) &
        (otpw_df['2_hour_trunc'] == F.date_trunc("hour", arrivals_shifted.ARR_TIME_PLUS_2H)),
        "left"
    )

    result = joined.groupBy("flight_id").agg(
        F.sum("ARR_DEL15").alias("dest_delayed_arrivals_2_4hr_before"),
        F.count("*").alias("dest_total_arrivals_2_4hr_before")
    )

In [0]:
# %skip
def compute_window_features_dest(otpw_df, lookback_start_hours = -4, lookback_end_hours = -2):
    '''Compute two window-based features from OTPW dataset:
    
        1. The number of delayed flights partitioned by destination airport in the specified lookback window from scheduled departure time.

        2. The percentange of delayed flights partitioned by destination airport in the specified lookback window from scheduled departure time.

    Default lookback window is 2-4 hours before departure. If there are no flights in the window, we count the # of flights as 0.
    '''
    # Repartition for performance
    otpw_df = otpw_df.repartition("DEST")

    # Define time-based window containing flights based on specified lookback hours before departure
    window_spec = Window.partitionBy("DEST") \
        .orderBy(F.col("sched_depart_date_time").cast("long")) \
        .rangeBetween(lookback_start_hours * 3600, lookback_end_hours * 3600) # can change these number for different window

    # The number of delayed flights partitioned by airport in the specified lookback window.
    otpw_df = otpw_df.withColumn(
        f"dest_delayed_flights_{-(lookback_end_hours)}_{-(lookback_start_hours)}hr_before",
        F.coalesce(
            F.sum("DEP_DEL15").over(window_spec),
            F.lit(0) # replace null with 0
        )
    )

    # The total number of flights partitioned by airport in the specified lookback window.
    otpw_df = otpw_df.withColumn(
        f"dest_total_flights_{-(lookback_end_hours)}_{-(lookback_start_hours)}hr_before",
        F.coalesce(
            F.count("*").over(window_spec),
            F.lit(0)
        )
    )

    # The percentange of delayed flights partitioned by airport in the specified lookback window.
    otpw_df = otpw_df.withColumn(
        f"dest_delay_pct_flights_{-(lookback_end_hours)}_{-(lookback_start_hours)}hr_before",
        F.coalesce(
            F.avg(F.coalesce(F.col("DEP_DEL15"), F.lit(0))).over(window_spec),
            F.lit(0) # replace null with 0
        )
    )

    return otpw_df

In [0]:
# %skip
# Compute destination congestion features for 2-4 window (ex. Count and % of delayed flights at destination airport 2-4 hours before departure from origin airport)
otpw_train_df = compute_window_features_dest(otpw_train_df)
otpw_test_df = compute_window_features_dest(otpw_test_df)

# Weather trend features
Instead of using a single weather snapshot, compute the change in weather over the past 6 hours at the origin airport. 

Ex: for visibility: subtract the observation from 6 hours before departure from the 2-hour-before observation to get `visibility_trend_6h` (positive = improving, negative = worsening), similarly for `windspeed_trend_6h`

In [0]:
def rejoin_otpw_and_weather(otpw_df, weather_df = None, hrs_before_departure = 2):
    '''Rejoin OTPW and Weather DF such that recorded weather observations are at least the specified number of hours before a flight's scheduled departure.
    Default # of hours is set to 2.
    '''
    if weather_df is None:
        weather_df = spark.read.parquet('dbfs:/mnt/mids-w261/datasets_final_project_2022/parquet_weather_data/')

    # filter on fm-15
    df_weather_filtered = weather_df.filter(col('REPORT_TYPE') == 'FM-15')   

    stations_set = {
        row["origin_station_id"]
        for row in otpw_df.select("origin_station_id").distinct().collect()
    }

    # filter on only stations in otpw
    df_weather_filtered = df_weather_filtered.filter(col('station').isin(stations_set))
    
    # Convert to timestamp. Create "ts" from "DATE"
    df_weather_filtered = df_weather_filtered.withColumn(
        "ts",
        to_timestamp(col("DATE"), "yyyy-MM-dd'T'HH:mm:ss")
    )

    df_weather_filtered = df_weather_filtered.filter(col("ts").isNotNull())

    # Truncate to hour. Create hour_trunc from "ts"
    df_weather_filtered = df_weather_filtered.withColumn(
        f"{hrs_before_departure}_hour_trunc",
        date_trunc("hour", col("ts"))
    )

    # List of columns to keep
    cols_to_keep = [
        'STATION',
        'DATE',
        'REPORT_TYPE',
        'SOURCE',
        'HourlyAltimeterSetting',
        'HourlyDewPointTemperature',
        'HourlyDryBulbTemperature',
        'HourlyPrecipitation',
        'HourlyPresentWeatherType',
        'HourlyPressureChange',
        'HourlyPressureTendency',
        'HourlyRelativeHumidity',
        'HourlySkyConditions',
        'HourlySeaLevelPressure',
        'HourlyStationPressure',
        'HourlyVisibility',
        'HourlyWetBulbTemperature',
        'HourlyWindDirection',
        'HourlyWindGustSpeed',
        'HourlyWindSpeed',
        f'{hrs_before_departure}_hour_trunc'
    ]

    # Filter the DataFrame to only these columns
    df_weather_filtered = df_weather_filtered.select(*cols_to_keep)

    # In df_weather_filtered, rename STATION → origin_station_id to match OTPW
    df_weather_filtered = df_weather_filtered.withColumnRenamed("STATION", "origin_station_id")

    weather_cols = [
        'HourlyAltimeterSetting',
        'HourlyDewPointTemperature',
        'HourlyDryBulbTemperature',
        'HourlyPrecipitation',
        'HourlyPresentWeatherType',
        'HourlyPressureChange',
        'HourlyPressureTendency',
        'HourlyRelativeHumidity',
        'HourlySkyConditions',
        'HourlySeaLevelPressure',
        'HourlyStationPressure',
        'HourlyVisibility',
        'HourlyWetBulbTemperature',
        'HourlyWindDirection',
        'HourlyWindGustSpeed',
        'HourlyWindSpeed'
    ]

    df_weather_renamed = df_weather_filtered
    for col_name in weather_cols:
        df_weather_renamed = df_weather_renamed.withColumnRenamed(col_name, f"{hrs_before_departure}h_{col_name}")
        print(f"Created {hrs_before_departure}h_{col_name}")
        assert(f'{hrs_before_departure}h_{col_name}' in df_weather_renamed.columns)

    # drop duplicates
    df_weather_renamed = df_weather_renamed.dropDuplicates(["origin_station_id", f'{hrs_before_departure}_hour_trunc'])

    # Drop columns that are no longer needed (original hourly weather vars and DATE)
    df_weather_renamed = df_weather_renamed.drop(*weather_cols)
    df_weather_renamed = df_weather_renamed.drop('DATE')
    otpw_df = otpw_df.drop(col('REPORT_TYPE'), col('SOURCE'))

    # Create "sched_depart_date_time" column, the scheduled flight departure in local time.
    if 'sched_depart_date_time' not in otpw_df.columns:
        print("Creating sched_depart_date_time in otpw")
        otpw_df = otpw_df.withColumn(
            "sched_depart_date_time",
            make_timestamp(
                col("YEAR"),             # year
                col("MONTH"),         # month
                col("DAY_OF_MONTH"),        # day
                substring(lpad(col("CRS_DEP_TIME"), 4, "0"), 1, 2),  # hour
                substring(lpad(col("CRS_DEP_TIME"), 4, "0"), 3, 2),  # minute
                col("sched_depart_date_time_UTC").substr(15, 2)      # seconds (usually "00")
            )
        )

    # Create "actual_depart_date_time" column, the actual flight departure in local time.
    if 'actual_depart_date_time' not in otpw_df.columns:
        print("Creating actual_depart_date_time in otpw")
        otpw_df = otpw_df.withColumn(
            "actual_depart_date_time",
            make_timestamp(
                col("YEAR"),             # year
                col("MONTH"),         # month
                col("DAY_OF_MONTH"),        # day
                substring(lpad(col("DEP_TIME"), 4, "0"), 1, 2),  # hour
                substring(lpad(col("DEP_TIME"), 4, "0"), 3, 2),  # minute
                lit("00")      # seconds (usually "00")
            )
        )

    # Optional safety
    otpw_df = otpw_df.filter(col("sched_depart_date_time").isNotNull())
    otpw_df = otpw_df.filter(col("actual_depart_date_time").isNotNull())
    otpw_df = otpw_df.withColumn(f"{hrs_before_departure}_hour_trunc", date_trunc("hour", least(col("sched_depart_date_time"), col("actual_depart_date_time")) - expr(f"INTERVAL {hrs_before_departure} HOURS")))

    otpw_df_joined = otpw_df.join(
        df_weather_renamed,
        on=["origin_station_id", f"{hrs_before_departure}_hour_trunc"],  # join keys
        how="left"
    )

    return otpw_df_joined

### Join OTPW and weather to match weather observations with flights 6 hours before departure

In [0]:
otpw_train_df = rejoin_otpw_and_weather(otpw_train_df, hrs_before_departure=6)
otpw_test_df = rejoin_otpw_and_weather(otpw_test_df, hrs_before_departure=6)

### Compute the weather trend features (subtract observations 6 hrs before departure with observations 2 hrs before)

In [0]:
# Did not include precipitation because most values are close to 0.
# If we include Visibility trend we need to remove visiblityCat to avoid multicollinearity
weather_features = [
  "DryBulbTemperature",
  "WindSpeed",
  "RelativeHumidity",
  "Visibility"
]

for feature in weather_features:
  otpw_train_df = otpw_train_df.withColumn(
    f"{feature}_trend_2h_6h",
    F.col(f'2h_Hourly{feature}') - F.col(f'6h_Hourly{feature}')
  )
  otpw_test_df = otpw_test_df.withColumn(
    f"{feature}_trend_2h_6h",
    F.col(f'2h_Hourly{feature}') - F.col(f'6h_Hourly{feature}')
  )
  print(f"{feature}_trend_2h_6h")

In [0]:
%skip
# Did not include precipitation because most values are close to 0.
# If we include Visibility trend we need to remove visiblityCat to avoid multicollinearity
weather_features = [
  "DryBulbTemperature_stdized",
  "WindSpeed_log_stdized",
  "RelativeHumidity_stdized",
  "Visibility"
]

for feature in weather_features:
  otpw_train_df = otpw_train_df.withColumn(
    f"{feature}_trend_6h",
    F.col(f'2h_Hourly{feature}') - F.col(f'6h_Hourly{feature}')
  )
  otpw_test_df = otpw_test_df.withColumn(
    f"{feature}_trend_6h",
    F.col(f'2h_Hourly{feature}') - F.col(f'6h_Hourly{feature}')
  )
  print(f"Created {feature}_trend_6h")

### Standardize weather trend features

In [0]:
def split_otpw_train_data(otpw_train_df):
    '''Splits train data by year.'''
    df_2015 = otpw_train_df.filter(
        (col("YEAR").cast("int") >= 2015) &
        (col("YEAR").cast("int") <= 2015)
    )
    df_2016 = otpw_train_df.filter(
        (col("YEAR").cast("int") >= 2016) &
        (col("YEAR").cast("int") <= 2016)
    )
    df_2017 = otpw_train_df.filter(
        (col("YEAR").cast("int") >= 2017) &
        (col("YEAR").cast("int") <= 2017)
    )
    df_2018 = otpw_train_df.filter(
        (col("YEAR").cast("int") >= 2018) &
        (col("YEAR").cast("int") <= 2018)
    )
    return df_2015, df_2016, df_2017, df_2018

def rejoin_otpw_train_splits(df_2015, df_2016, df_2017, df_2018):
    '''Rejoins the train data by year back into a single dataset'''
    return reduce(DataFrame.union, [df_2015, df_2016, df_2017, df_2018])
  
def standardize_features(otpw_data, features = [], stats = None):
    # Compute stats in one aggregation
    agg_exprs = [mean(c).alias(f"{c}_mean") for c in features] + \
                [stddev(c).alias(f"{c}_std") for c in features]

    if stats is None:
        stats = otpw_data.agg(*agg_exprs).first()

    # Apply standardization to each feature using the current fold's statistics
    for c in features:
        mu = stats[f"{c}_mean"]
        sd = stats[f"{c}_std"]
        otpw_data = otpw_data.withColumn(
            f"{c}_stdized", 
            (col(c) - mu) / sd
        )
    return otpw_data

def get_mean_and_std(otpw_data, features):
    # Compute stats in one aggregation
    agg_exprs = [mean(c).alias(f"{c}_mean") for c in features] + \
                [stddev(c).alias(f"{c}_std") for c in features]

    stats = otpw_data.agg(*agg_exprs).first()

    return stats

In [0]:
weather_trend_features = [
  "DryBulbTemperature_trend_2h_6h",
  "RelativeHumidity_trend_2h_6h",
  "Visibility_trend_2h_6h"
]

otpw_2015_df, otpw_2016_df, otpw_2017_df, otpw_2018_df = split_otpw_train_data(otpw_train_df)

otpw_2015_df = standardize_features(otpw_2015_df, weather_trend_features)
otpw_2016_df = standardize_features(otpw_2016_df, weather_trend_features)
otpw_2017_df = standardize_features(otpw_2017_df, weather_trend_features)
otpw_2018_df = standardize_features(otpw_2018_df, weather_trend_features)

otpw_train_df = rejoin_otpw_train_splits(otpw_2015_df, otpw_2016_df, otpw_2017_df, otpw_2018_df)

# Get entire train stats to use for standardizing test set
otpw_train_stats = get_mean_and_std(otpw_train_df, weather_trend_features)
otpw_test_df = standardize_features(otpw_test_df, weather_trend_features, stats = otpw_train_stats)

### Log wind speed before standardizing

In [0]:
# Log-transform (creates a new column)
otpw_train_df = otpw_train_df.withColumn("WindSpeed_trend_2h_6h_log", log1p(col("WindSpeed_trend_2h_6h")))

# Split train data by year again before standardizing features
otpw_2015_df, otpw_2016_df, otpw_2017_df, otpw_2018_df = split_otpw_train_data(otpw_train_df)

otpw_2015_df = standardize_features(otpw_2015_df, features = ["WindSpeed_trend_2h_6h_log"])
otpw_2016_df = standardize_features(otpw_2016_df, features = ["WindSpeed_trend_2h_6h_log"])
otpw_2017_df = standardize_features(otpw_2017_df, features = ["WindSpeed_trend_2h_6h_log"])
otpw_2018_df = standardize_features(otpw_2018_df, features = ["WindSpeed_trend_2h_6h_log"])

otpw_train_df = rejoin_otpw_train_splits(otpw_2015_df, otpw_2016_df, otpw_2017_df, otpw_2018_df)
otpw_train_stats = get_mean_and_std(otpw_train_df, features = ["WindSpeed_trend_2h_6h_log"])

otpw_test_df = otpw_test_df.withColumn("WindSpeed_trend_2h_6h_log", log1p(col("WindSpeed_trend_2h_6h")))
otpw_test_df = standardize_features(otpw_test_df, features = ['WindSpeed_trend_2h_6h_log'], stats = otpw_train_stats)

In [0]:
for col in ['PrecipBinary',
'DryBulbTemperature_trend_2h_6h_stdized',
'WindSpeed_trend_2h_6h_log_stdized',
'RelativeHumidity_trend_2h_6h_stdized',
'Visibility_trend_2h_6h_stdized']:
    print(col in otpw_test_df.columns)

In [0]:
%skip
weather_features_6h = [
  "6h_HourlyDryBulbTemperature",
  "6h_HourlyRelativeHumidity",
  "6h_HourlyVisibility"
]

otpw_2015_df, otpw_2016_df, otpw_2017_df, otpw_2018_df = split_otpw_train_data(otpw_train_df)

otpw_2015_df = standardize_features(otpw_2015_df, weather_features_6h)
otpw_2016_df = standardize_features(otpw_2016_df, weather_features_6h)
otpw_2017_df = standardize_features(otpw_2017_df, weather_features_6h)
otpw_2018_df = standardize_features(otpw_2018_df, weather_features_6h)

otpw_train_df = rejoin_otpw_train_splits(otpw_2015_df, otpw_2016_df, otpw_2017_df, otpw_2018_df)

# Get entire train stats to use for standardizing test set
otpw_train_stats = get_mean_and_std(otpw_train_df, weather_features_6h)
otpw_test_df = standardize_features(otpw_test_df, weather_features_6h, stats = otpw_train_stats)

# Tail number delay propagation
For each flight, look up the same aircraft's previous leg using `TAIL_NUM` and check if that prior flight was delayed. That's the propagation effect that's going to be very powerful.

Create two features: 

1. `prev_leg_delayed` (binary: was the previous flight on this aircraft delayed?) 

2. `prev_leg_delay_minutes` (how many minutes was it delayed?). This captures the cascading effect: if an aircraft arrived late from its previous route, it's likely to depart late on its next one. 

We could sort by `TAIL_NUM` and `sched_depart_date_time`, then use a lag window to get the prior flight's `DEP_DEL15` and `DEP_DELAY`. Just note, we should be careful about the leakage. 

In [0]:
display(otpw_train_df)

# Cyclic time encoding

Right now we are just using OHE. Should we consider doing sin/cosin to make 0 and 23 hour and Jan / Dec close to each other?

# Final touchups to train and test sets before modeling

In [0]:
# Add DEP_HOUR column from sched_depart_date_time
otpw_train_df = otpw_train_df.withColumn("DEP_HOUR", hour("sched_depart_date_time"))
otpw_test_df = otpw_test_df.withColumn("DEP_HOUR", hour('sched_depart_date_time'))

# Cast DISTANCE to double
otpw_train_df = otpw_train_df.withColumn("DISTANCE", otpw_train_df["DISTANCE"].cast("double"))
otpw_test_df = otpw_test_df.withColumn("DISTANCE", otpw_test_df["DISTANCE"].cast("double"))

In [0]:
# Write data to parquet
otpw_train_df.write.mode("overwrite").parquet(f"{TEAM_DIR}/041326_otpw_train_df_without_tail_prop.parquet")
otpw_test_df.write.mode("overwrite").parquet(f"{TEAM_DIR}/041326_otpw_test_df_without_tail_prop.parquet")

# otpw_train_df = spark.read.parquet(f"{TEAM_DIR}/041326_otpw_train_df.parquet")
# otpw_test_df = spark.read.parquet(f"{TEAM_DIR}/041326_otpw_test_df.parquet")

In [0]:
otpw_train_df = spark.read.parquet(f"{TEAM_DIR}/041326_otpw_train_df.parquet")
otpw_test_df = spark.read.parquet(f"{TEAM_DIR}/041326_otpw_test_df.parquet")

display(otpw_test_df)

# tail number delay propagation

In [0]:
otpw_train_df = spark.read.parquet(f"{TEAM_DIR}/041326_otpw_train_df.parquet")

In [0]:
# ----------------------------
# 1. FL_DATE as date
# ----------------------------
otpw_train_df = otpw_train_df.withColumn("FL_DATE_TS", F.to_date("FL_DATE"))

# ----------------------------
# 2. Convert HHMM → minutes since midnight
# ----------------------------
otpw_train_df = otpw_train_df.withColumn(
    "dep_min",
    (F.col("CRS_DEP_TIME") / 100).cast("int") * 60 + (F.col("CRS_DEP_TIME") % 100)
).withColumn(
    "arr_min",
    (F.col("CRS_ARR_TIME") / 100).cast("int") * 60 + (F.col("CRS_ARR_TIME") % 100)
)

# ----------------------------
# 3. Detect overnight flight
# ----------------------------
otpw_train_df = otpw_train_df.withColumn(
    "day_offset",
    F.when(F.col("arr_min") < F.col("dep_min"), 1).otherwise(0)
)

# ----------------------------
# 4. Convert everything to seconds since epoch
# ----------------------------
otpw_train_df = otpw_train_df.withColumn(
    "base_ts",
    F.unix_timestamp("FL_DATE_TS")
).withColumn(
    "crs_dep_datetime",
    F.to_timestamp(F.col("base_ts") + F.col("dep_min") * 60)
).withColumn(
    "crs_arr_datetime",
    F.to_timestamp(F.col("base_ts") + F.col("arr_min") * 60 + F.col("day_offset") * 86400)
)

In [0]:
display(otpw_train_df)

In [0]:
window_tail = Window.partitionBy("TAIL_NUM") \
                    .orderBy("sched_depart_date_time")

# get the previous delay time and if the flight was delayed
otpw_train_df = otpw_train_df.withColumn(
    "prev_DEP_DEL15",
    F.lag("DEP_DEL15", 1).over(window_tail)
).withColumn(
    "prev_DEP_DELAY",
    F.lag("DEP_DELAY", 1).over(window_tail)
)

In [0]:
# turnaround time

# ----------------------------------------
# NEW: previous arrival time
# ----------------------------------------
otpw_train_df = otpw_train_df.withColumn(
    "prev_arrival_ts",
    F.lag("crs_arr_datetime", 1).over(window_tail)
)

# ----------------------------------------
# NEW: time between flights (turnaround)
# ----------------------------------------
otpw_train_df = otpw_train_df.withColumn(
    "turnaround_minutes",
    (F.col("sched_depart_date_time").cast("long") -
     F.col("prev_arrival_ts").cast("long")) / 60
)

In [0]:
display(otpw_train_df)